In [1]:
import json
import os
import re
import subprocess
from pathlib import Path
from urllib.parse import urlparse

try:
    import pandas as pd
    PANDAS_AVAILABLE = True
except ImportError:
    PANDAS_AVAILABLE = False

In [2]:
class FootprintTransparencyChecker:
    """
    FootprintTransparencyChecker

    This checker evaluates whether a research software artifact provides
    transparent information about its execution footprint.

    It checks for documentation and configuration evidence related to:
    - runtime requirements
    - hardware requirements
    - operating system/platform constraints
    - memory/CPU/GPU requirements
    - dependency specifications
    - installation/setup instructions
    - environment/container files

    Formal idea:
        footprintTransparency : A → {True, False}

    The artifact passes if enough footprint-related evidence is found.
    """

    def __init__(
        self,
        json_file,
        download_dir="downloads",
        minimum_score=4
    ):
        self.json_file = json_file
        self.download_dir = Path(download_dir)
        self.download_dir.mkdir(parents=True, exist_ok=True)

        self.minimum_score = minimum_score
        self.artifacts = self.load_metadata(json_file)
        self.results = []

    def load_metadata(self, json_file):
        with open(json_file, "r", encoding="utf-8") as file:
            data = json.load(file)

        return data.get("artifacts", {})

    def is_git_repository(self, uri):
        return isinstance(uri, str) and uri.startswith("https://github.com/")

    def repo_name_from_uri(self, uri):
        parsed = urlparse(uri)
        repo_name = parsed.path.rstrip("/").split("/")[-1]

        if repo_name.endswith(".git"):
            repo_name = repo_name[:-4]

        return repo_name or "repository"

    def clone_repository(self, artifact_id, uri):
        repo_name = self.repo_name_from_uri(uri)
        target_dir = self.download_dir / repo_name

        if target_dir.exists():
            print(f"📁 Repository already exists: {target_dir}")
            return target_dir

        print(f"⬇️ Cloning repository: {uri}")

        try:
            result = subprocess.run(
                ["git", "clone", "--depth", "1", uri, str(target_dir)],
                stdout=subprocess.PIPE,
                stderr=subprocess.PIPE,
                text=True,
                timeout=120
            )

            if result.returncode != 0:
                print(f"❌ Failed to clone repository for {artifact_id}")
                print(result.stderr.strip())
                return None

            print(f"✅ Cloned to: {target_dir}")
            return target_dir

        except Exception as e:
            print(f"❌ Clone error for {artifact_id}: {e}")
            return None

    def find_files_by_names(self, repo_dir, names):
        matches = []

        for root, dirs, files in os.walk(repo_dir):
            for file in files:
                lower_file = file.lower()

                if lower_file in names:
                    matches.append(Path(root) / file)

        return matches

    def find_files_by_extensions(self, repo_dir, extensions):
        matches = []

        for root, dirs, files in os.walk(repo_dir):
            for file in files:
                if any(file.lower().endswith(ext) for ext in extensions):
                    matches.append(Path(root) / file)

        return matches

    def read_text_file(self, path, max_chars=300000):
        try:
            with open(path, "r", encoding="utf-8", errors="ignore") as file:
                return file.read(max_chars)
        except Exception:
            return ""

    def collect_documentation_text(self, repo_dir):
        documentation_names = {
            "readme.md",
            "readme.rst",
            "readme.txt",
            "installation.md",
            "install.md",
            "setup.md",
            "usage.md",
            "requirements.md",
            "environment.md",
            "docker.md",
            "docs.md",
            "contributing.md"
        }

        documentation_files = self.find_files_by_names(repo_dir, documentation_names)

        docs_dir = repo_dir / "docs"
        if docs_dir.exists():
            documentation_files.extend(
                self.find_files_by_extensions(docs_dir, [".md", ".rst", ".txt"])
            )

        unique_files = []
        seen = set()

        for path in documentation_files:
            resolved = str(path.resolve())
            if resolved not in seen:
                unique_files.append(path)
                seen.add(resolved)

        combined_text = ""

        for path in unique_files:
            combined_text += f"\n\n--- FILE: {path.relative_to(repo_dir)} ---\n\n"
            combined_text += self.read_text_file(path)

        return combined_text, unique_files

    def file_exists(self, repo_dir, possible_paths):
        for relative_path in possible_paths:
            if (repo_dir / relative_path).exists():
                return True, relative_path

        return False, None

    def count_declared_dependencies(self, repo_dir):
        dependency_files = [
            "requirements.txt",
            "environment.yml",
            "environment.yaml",
            "pyproject.toml",
            "setup.py",
            "Pipfile",
            "poetry.lock",
            "package.json",
            "renv.lock",
            "DESCRIPTION"
        ]

        found_files = []

        for file in dependency_files:
            if (repo_dir / file).exists():
                found_files.append(file)

        dependency_count = 0

        requirements_path = repo_dir / "requirements.txt"
        if requirements_path.exists():
            text = self.read_text_file(requirements_path)
            for line in text.splitlines():
                line = line.strip()
                if line and not line.startswith("#") and not line.startswith("-"):
                    dependency_count += 1

        environment_path = repo_dir / "environment.yml"
        if environment_path.exists():
            text = self.read_text_file(environment_path)
            dependency_count += len([
                line for line in text.splitlines()
                if line.strip().startswith("- ")
            ])

        pyproject_path = repo_dir / "pyproject.toml"
        if pyproject_path.exists():
            text = self.read_text_file(pyproject_path)
            dependency_count += len(re.findall(r'"[^"]+>=?[^"]*"', text))

        return dependency_count, found_files

    def check_keywords(self, text, keywords):
        text_lower = text.lower()
        found = []

        for keyword in keywords:
            if keyword.lower() in text_lower:
                found.append(keyword)

        return found

    def evaluate_footprint_transparency(self, repo_dir):
        documentation_text, documentation_files = self.collect_documentation_text(repo_dir)
        dependency_count, dependency_files = self.count_declared_dependencies(repo_dir)

        text_lower = documentation_text.lower()

        runtime_keywords = [
            "python", "runtime", "run", "execute", "execution",
            "installation", "install", "setup", "usage"
        ]

        hardware_keywords = [
            "hardware", "cpu", "processor", "memory", "ram",
            "gpu", "cuda", "disk", "storage"
        ]

        platform_keywords = [
            "windows", "linux", "macos", "ubuntu", "operating system",
            "os", "platform", "docker", "container"
        ]

        environment_keywords = [
            "requirements.txt", "environment.yml", "conda",
            "pip install", "virtualenv", "venv", "poetry",
            "dependencies", "dependency"
        ]

        performance_keywords = [
            "performance", "benchmark", "runtime", "time",
            "seconds", "minutes", "memory usage", "resource"
        ]

        runtime_evidence = self.check_keywords(documentation_text, runtime_keywords)
        hardware_evidence = self.check_keywords(documentation_text, hardware_keywords)
        platform_evidence = self.check_keywords(documentation_text, platform_keywords)
        environment_evidence = self.check_keywords(documentation_text, environment_keywords)
        performance_evidence = self.check_keywords(documentation_text, performance_keywords)

        has_readme, readme_path = self.file_exists(
            repo_dir,
            ["README.md", "README.rst", "README.txt", "readme.md", "readme.rst", "readme.txt"]
        )

        has_requirements, requirements_path = self.file_exists(
            repo_dir,
            ["requirements.txt", "environment.yml", "environment.yaml", "pyproject.toml", "setup.py", "Pipfile", "package.json"]
        )

        has_container, container_path = self.file_exists(
            repo_dir,
            ["Dockerfile", "docker-compose.yml", "docker-compose.yaml", ".devcontainer/devcontainer.json"]
        )

        has_ci, ci_path = self.file_exists(
            repo_dir,
            [".github/workflows", ".gitlab-ci.yml", "azure-pipelines.yml"]
        )

        score = 0
        evidence = []

        if has_readme:
            score += 1
            evidence.append(f"README present: {readme_path}")

        if has_requirements:
            score += 1
            evidence.append(f"Dependency/environment file present: {requirements_path}")

        if has_container:
            score += 1
            evidence.append(f"Container/deployment file present: {container_path}")

        if runtime_evidence:
            score += 1
            evidence.append(f"Runtime/setup documentation terms found: {', '.join(runtime_evidence[:8])}")

        if environment_evidence:
            score += 1
            evidence.append(f"Environment/dependency documentation terms found: {', '.join(environment_evidence[:8])}")

        if platform_evidence:
            score += 1
            evidence.append(f"Platform documentation terms found: {', '.join(platform_evidence[:8])}")

        if hardware_evidence:
            score += 1
            evidence.append(f"Hardware/resource documentation terms found: {', '.join(hardware_evidence[:8])}")

        if performance_evidence:
            score += 1
            evidence.append(f"Performance/resource documentation terms found: {', '.join(performance_evidence[:8])}")

        if has_ci:
            score += 1
            evidence.append(f"CI/workflow evidence present: {ci_path}")

        footprint_transparent = score >= self.minimum_score

        missing = []

        if not has_readme:
            missing.append("README or equivalent documentation")

        if not has_requirements:
            missing.append("dependency/environment specification")

        if not runtime_evidence:
            missing.append("runtime/setup information")

        if not hardware_evidence:
            missing.append("hardware/resource information")

        if not platform_evidence:
            missing.append("platform/OS/container information")

        if not performance_evidence:
            missing.append("performance/resource-footprint information")

        return {
            "footprint_transparent": footprint_transparent,
            "score": score,
            "minimum_score": self.minimum_score,
            "documentation_files_count": len(documentation_files),
            "documentation_files": [str(path.relative_to(repo_dir)) for path in documentation_files[:20]],
            "dependency_files": dependency_files,
            "dependency_count": dependency_count,
            "has_readme": has_readme,
            "has_requirements": has_requirements,
            "has_container": has_container,
            "has_ci": has_ci,
            "runtime_evidence": runtime_evidence,
            "hardware_evidence": hardware_evidence,
            "platform_evidence": platform_evidence,
            "environment_evidence": environment_evidence,
            "performance_evidence": performance_evidence,
            "evidence": evidence,
            "missing": missing
        }

    def check_artifact(self, artifact_id, artifact_data):
        title = artifact_data.get("title", "")
        uri = artifact_data.get("uri", "")

        print("\n" + "=" * 80)
        print(f"🔍 Footprint Transparency Check for {artifact_id}")
        print(f"📦 Title: {title}")
        print(f"🔗 URI: {uri}")

        artifact_result = {
            "artifact_id": artifact_id,
            "title": title,
            "uri": uri,
            "footprint_transparent": False,
            "status": "failed"
        }

        if not self.is_git_repository(uri):
            print("❌ Unsupported artifact type for this checker.")
            artifact_result["reason"] = "Unsupported artifact type."
            return artifact_result

        repo_dir = self.clone_repository(artifact_id, uri)

        if repo_dir is None:
            artifact_result["reason"] = "Repository could not be cloned."
            artifact_result["status"] = "not_evaluated_repository_unavailable"
            return artifact_result

        result = self.evaluate_footprint_transparency(repo_dir)
        artifact_result.update(result)

        print("\n📊 Footprint transparency evidence:")
        print(f" - Score: {result['score']} / required {result['minimum_score']}")
        print(f" - Documentation files found: {result['documentation_files_count']}")
        print(f" - Dependency/environment files: {', '.join(result['dependency_files']) if result['dependency_files'] else 'None'}")
        print(f" - README present: {'✅' if result['has_readme'] else '❌'}")
        print(f" - Requirements/environment spec present: {'✅' if result['has_requirements'] else '❌'}")
        print(f" - Container/deployment file present: {'✅' if result['has_container'] else '❌'}")
        print(f" - CI/workflow evidence present: {'✅' if result['has_ci'] else '❌'}")

        print("\n🔎 Evidence found:")
        if result["evidence"]:
            for item in result["evidence"]:
                print(f" - {item}")
        else:
            print(" - No footprint evidence found.")

        if result["missing"]:
            print("\n⚠️ Missing or weak evidence:")
            for item in result["missing"]:
                print(f" - {item}")

        if result["footprint_transparent"]:
            artifact_result["status"] = "passed"
            print("\n✅ Footprint Transparency Result: PASSED")
        else:
            artifact_result["status"] = "failed"
            print("\n❌ Footprint Transparency Result: FAILED")

        return artifact_result

    def run(self):
        self.results = []

        print("🌱 Starting Footprint Transparency Fitness Function")
        print(f"📄 Metadata file: {self.json_file}")
        print(f"📁 Download directory: {self.download_dir}")

        for artifact_id, artifact_data in self.artifacts.items():
            result = self.check_artifact(artifact_id, artifact_data)
            self.results.append(result)

        print("\n" + "=" * 80)
        print("📌 Footprint Transparency Summary")
        print("=" * 80)

        for result in self.results:
            icon = "✅" if result["footprint_transparent"] else "❌"
            print(f"{icon} {result['artifact_id']}: {result['status']}")

        return self.results

In [3]:
checker = FootprintTransparencyChecker(
    json_file="artifacts.json",
    download_dir="downloads",
    minimum_score=4
)

footprint_results = checker.run()

🌱 Starting Footprint Transparency Fitness Function
📄 Metadata file: artifacts.json
📁 Download directory: downloads

🔍 Footprint Transparency Check for artifact_1
📦 Title: We provide our resources in a dedicated repository
🔗 URI: https://github.com/hihey54/hicss58
📁 Repository already exists: downloads/hicss58

📊 Footprint transparency evidence:
 - Score: 4 / required 4
 - Documentation files found: 2
 - Dependency/environment files: requirements.txt
 - README present: ✅
 - Requirements/environment spec present: ✅
 - Container/deployment file present: ❌
 - CI/workflow evidence present: ❌

🔎 Evidence found:
 - README present: README.md
 - Dependency/environment file present: requirements.txt
 - Platform documentation terms found: os
 - Performance/resource documentation terms found: resource

⚠️ Missing or weak evidence:
 - runtime/setup information
 - hardware/resource information

✅ Footprint Transparency Result: PASSED

🔍 Footprint Transparency Check for artifact_2
📦 Title: Trending C

In [4]:
if PANDAS_AVAILABLE:
    df = pd.DataFrame(footprint_results)

    columns_to_show = [
        "artifact_id",
        "title",
        "footprint_transparent",
        "status",
        "score",
        "minimum_score",
        "documentation_files_count",
        "dependency_count",
        "has_readme",
        "has_requirements",
        "has_container",
        "has_ci"
    ]

    existing_columns = [col for col in columns_to_show if col in df.columns]
    display(df[existing_columns])
else:
    for result in footprint_results:
        print(result)

,artifact_id,title,footprint_transparent,status,score,minimum_score,documentation_files_count,dependency_count,has_readme,has_requirements,has_container,has_ci
0,artifact_1,We provide our resources in a dedicated reposi...,True,passed,4.0,4.0,2.0,3.0,True,True,False,False
1,artifact_2,Trending Customer Dataset,False,not_evaluated_repository_unavailable,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,artifact_3,Python algorithms,True,passed,9.0,4.0,23.0,28.0,True,True,True,True
3,artifact_4,Scikit-learn,True,passed,9.0,4.0,43.0,59.0,True,True,True,True
4,artifact_5,Pandas,True,passed,8.0,4.0,5.0,101.0,True,True,False,True
5,artifact_6,NumPy,True,passed,9.0,4.0,8.0,42.0,True,True,True,True
6,artifact_7,Matplotlib,True,passed,8.0,4.0,12.0,94.0,True,True,True,True
7,artifact_8,Scrapy,True,passed,8.0,4.0,57.0,19.0,True,True,False,True
8,artifact_9,Flask,True,passed,9.0,4.0,81.0,9.0,True,True,True,True
9,artifact_10,TensorFlow,True,passed,7.0,4.0,207.0,0.0,True,False,False,True


In [5]:
output_file = "footprint_transparency_results.json"

with open(output_file, "w", encoding="utf-8") as file:
    json.dump(footprint_results, file, indent=4)

print(f"✅ Results saved to {output_file}")

✅ Results saved to footprint_transparency_results.json
